# Modelos de Machine Learning – Previsão de IPCA

Este notebook implementa modelos de ML para prever:

- IPCA mensal (t+1)
- IPCA acumulado em 12 meses (t+12)

Usando:
- SELIC e derivados (NB5)
- Blocos macroeconômicos (NB6)
- Curva de Phillips (NB7)
- Modelos estatísticos (NB8)

Compatível com Jupytext.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
import shap

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

pd.set_option("display.float_format", "{:.4f}".format)
print("PYTHONPATH atualizado:", ROOT)

# 1. Carregar datasets processados

In [ ]:
path_selic = os.path.join(ROOT, "data", "processed", "ipca_selic_features.parquet")
path_blocos = os.path.join(ROOT, "data", "processed", "ipca_blocos_macro.parquet")
path_phillips = os.path.join(ROOT, "data", "processed", "curva_phillips_features.parquet")

df_selic = pd.read_parquet(path_selic)
df_blocos = pd.read_parquet(path_blocos)
df_phillips = pd.read_parquet(path_phillips)

df_selic.index = pd.to_datetime(df_selic.index)
df_blocos.index = pd.to_datetime(df_blocos.index)
df_phillips.index = pd.to_datetime(df_phillips.index)

df_selic.tail(), df_blocos.tail(), df_phillips.tail()

# 2. Base consolidada

In [ ]:
df = df_selic.join(df_blocos, how="inner").join(df_phillips, how="inner")
df = df.sort_index()
df.tail()

# 3. Criar targets de previsão

- IPCA t+1
- IPCA 12m t+12

In [ ]:
df["ipca_t"] = df["ipca"]
df["ipca_t_plus_1"] = df["ipca"].shift(-1)
df["ipca_12m_t_plus_12"] = df["ipca_12m"].shift(-12)

df.tail()

# 4. Engenharia de features

- normalização
- seleção automática
- rolling windows
- interações

In [ ]:
# Rolling windows
df["ipca_rolling_3"] = df["ipca"].rolling(3).mean()
df["ipca_rolling_6"] = df["ipca"].rolling(6).mean()
df["ipca_rolling_12"] = df["ipca"].rolling(12).mean()

# Interações
df["selic_x_hiato"] = df["selic_real"] * df["hiato_proxy"]

df = df.dropna()
df.tail()

# 5. Separar treino e teste (TimeSeriesSplit)

In [ ]:
target_short = "ipca_t_plus_1"
target_med = "ipca_12m_t_plus_12"

features = [c for c in df.columns if c not in ["ipca_t_plus_1", "ipca_12m_t_plus_12"]]

X = df[features]
y_short = df[target_short]
y_med = df[target_med]

split = TimeSeriesSplit(n_splits=5)

# 6. Normalização

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 7. Modelos de Machine Learning

- Linear Regression
- Ridge
- Lasso
- Random Forest
- Extra Trees
- Gradient Boosting
- XGBoost

In [ ]:
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001),
    "RandomForest": RandomForestRegressor(n_estimators=500),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=500),
    "GradientBoosting": GradientBoostingRegressor(),
    "XGBoost": XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
    ),
}

# 8. Função de avaliação

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, pred, squared=False)
    mae = mean_absolute_error(y_test, pred)
    return rmse, mae, pred

# 9. Avaliação – Previsão de IPCA t+1

In [ ]:
results_short = {}

train_size = int(len(df) * 0.8)

X_train = X_scaled[:train_size]
X_test = X_scaled[train_size:]
y_train = y_short[:train_size]
y_test = y_short[train_size:]

for name, model in models.items():
    rmse, mae, pred = evaluate_model(model, X_train, y_train, X_test, y_test)
    results_short[name] = {"rmse": rmse, "mae": mae, "pred": pred}

results_short

# 10. Ranking dos modelos (t+1)

In [ ]:
df_rank_short = pd.DataFrame(results_short).T.sort_values("rmse")
df_rank_short

# 11. Avaliação – Previsão de IPCA 12m à frente

In [ ]:
results_med = {}

y_train_med = y_med[:train_size]
y_test_med = y_med[train_size:]

for name, model in models.items():
    rmse, mae, pred = evaluate_model(model, X_train, y_train_med, X_test, y_test_med)
    results_med[name] = {"rmse": rmse, "mae": mae, "pred": pred}

df_rank_med = pd.DataFrame(results_med).T.sort_values("rmse")
df_rank_med

# 12. SHAP – Explicabilidade do melhor modelo

In [ ]:
best_model_name = df_rank_short.index[0]
best_model = models[best_model_name]

best_model.fit(X_train, y_train)

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_train)

shap.summary_plot(shap_values, X_train, feature_names=features)

# 13. Exportar previsões

In [ ]:
df_pred = pd.DataFrame(
    {
        "y_test_short": y_test,
        "y_pred_short": results_short[best_model_name]["pred"],
        "y_test_med": y_test_med,
        "y_pred_med": results_med[best_model_name]["pred"],
    },
    index=df.index[train_size:],
)

output_path = os.path.join(ROOT, "data", "processed", "ml_predictions.parquet")
df_pred.to_parquet(output_path)

print("Previsões salvas em:", output_path)

# 14. Conclusões

- Modelos de árvore e boosting tendem a superar modelos lineares.
- XGBoost e ExtraTrees geralmente são os melhores para inflação.
- SHAP mostra quais variáveis realmente importam.
- Este notebook prepara o terreno para:
  - NB10 (Dashboard)
  - NB11 (Relatório final)